# AmsiNet - Step 20: numbers for the yellow placeholders (CPU, about 5 minutes)
Run on a **CPU runtime** (Runtime > Change runtime type > CPU), then *Run all* and paste the output back. It prints: (1) the CPU model, (2) parameters of the three flat/word classifiers with a 10-class head, (3) median single-image CPU latency, (4) the MDIW-13 split counts used in the paper.

In [ ]:
import platform, subprocess, time, os, json, numpy as np, pandas as pd
try:
    print('CPU model:', [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')][0])
except Exception as e: print('cpuinfo failed', e)
print('logical CPUs:', os.cpu_count(), '| platform:', platform.platform())
import torch
print('torch', torch.__version__, '| threads', torch.get_num_threads())

In [ ]:
try: import timm
except ImportError:
    !pip -q install timm
    import timm
C = 10
MODELS = {'EfficientNet-B0 (AmsiNet-E word agent)': 'efficientnet_b0.ra_in1k',
          'ViT-S/16 (flat)': 'vit_small_patch16_224.augreg_in21k',
          'ViT-B/16 (flat)': 'vit_base_patch16_224.augreg_in21k'}
rows = []
torch.set_grad_enabled(False)
for name, mn in MODELS.items():
    m = timm.create_model(mn, pretrained=False, num_classes=C).eval()
    params = sum(p.numel() for p in m.parameters()) / 1e6
    x = torch.randn(1, 3, 224, 224)
    for _ in range(5): m(x)
    ts = []
    for _ in range(30):
        t0 = time.perf_counter(); m(x); ts.append((time.perf_counter() - t0) * 1000)
    rows.append((name, round(params, 2), round(float(np.median(ts)), 1)))
R = pd.DataFrame(rows, columns=['model', 'params (M)', 'median CPU latency (ms), batch 1'])
display(R)

In [ ]:
from google.colab import drive
import glob
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/amsinet'
print(sorted(os.listdir(f'{ROOT}/mdiw')))
for f in sorted(glob.glob(f'{ROOT}/mdiw/*.csv')):
    try: W = pd.read_csv(f)
    except Exception as e: print('skip', f, e); continue
    if 'split' not in W.columns: continue
    print('=====', os.path.basename(f), '| rows', len(W), '| columns', list(W.columns))
    agg = {'rows': ('split', 'size')}
    for col in ('doc', 'document', 'doc_id', 'line', 'line_id'):
        if col in W.columns: agg[col + ' (unique)'] = (col, 'nunique')
    display(W.groupby('split').agg(**agg))
    if 'script' in W.columns: display(W.groupby(['script', 'split']).size().unstack(fill_value=0))